Import Libraries

In [2]:
import pandas as pd
import re
import string
import emoji
import nltk
import spacy

In [38]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sabri\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\sabri\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

Load Dataset

In [3]:
df = pd.read_csv("../data/raw_data/mae_reviews.csv")

Exploratory Data Analysis (EDA)

In [ ]:
## 3.1 Dataset Overview
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)

print(f"Number of rows    : {df.shape[0]}")
print(f"Number of columns : {df.shape[1]}")

print("\nColumn Names:")
print(df.columns.tolist())

DATASET OVERVIEW
Number of rows    : 4155
Number of columns : 5

Column Names:
['review_id', 'review', 'rating', 'review_date', 'app_version']


In [ ]:
## 3.2 Dataset Information
print("=" * 50)
print("DATASET INFORMATION")
print("=" * 50)

df.info()

DATASET INFORMATION
<class 'pandas.DataFrame'>
RangeIndex: 4155 entries, 0 to 4154
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   review_id    4155 non-null   str  
 1   review       4155 non-null   str  
 2   rating       4155 non-null   int64
 3   review_date  4155 non-null   str  
 4   app_version  3428 non-null   str  
dtypes: int64(1), str(4)
memory usage: 863.8 KB


In [ ]:
## 3.3 Missing Value Analysis
print("=" * 50)
print("MISSING VALUE ANALYSIS")
print("=" * 50)

missing_values = df.isnull().sum()

missing_percentage = (missing_values / len(df)) * 100

missing_df = pd.DataFrame({
    "Missing Count": missing_values,
    "Missing Percentage (%)": missing_percentage.round(2)
})

missing_df

MISSING VALUE ANALYSIS


,Missing Count,Missing Percentage (%)
review_id,0,0.0
review,0,0.0
rating,0,0.0
review_date,0,0.0
app_version,727,17.5


In [ ]:
## 3.4 Duplicate Value Analysis
print("=" * 50)
print("DUPLICATE VALUE ANALYSIS")
print("=" * 50)

duplicate_reviews = df.duplicated(subset=["review"]).sum()

print(f"Duplicate Reviews : {duplicate_reviews}")

DUPLICATE VALUE ANALYSIS
Duplicate Reviews : 0


In [8]:
## 3.5 Rating Distribution
print("=" * 50)
print("RATING DISTRIBUTION")
print("=" * 50)

rating_counts = df["rating"].value_counts().sort_index()

print(rating_counts)

RATING DISTRIBUTION
rating
1    1881
2     341
3     319
4     185
5    1429
Name: count, dtype: int64


NLP Preprocessing

In [10]:
import re
import string
import emoji
import nltk
import spacy

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [11]:
nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sabri\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sabri\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [12]:
nlp = spacy.load("en_core_web_sm")

In [13]:
df["clean_review"] = df["review"]

Convert Text to lowercase

In [15]:
# Convert all text to lowercase
df["clean_review"] = df["clean_review"].str.lower()
df[["review", "clean_review"]].head()

,review,clean_review
0,A dependable banking app that makes handling m...,a dependable banking app that makes handling m...
1,very good app,very good app
2,Amazing service and great experience! 😊 Thank ...,amazing service and great experience! 😊 thank ...
3,I gave this app a chance it works up until the...,i gave this app a chance it works up until the...
4,"Can't reload my phone credit through FPX, noti...","can't reload my phone credit through fpx, noti..."


Remove URL from Text

In [16]:
# Function to remove URLs from text
def remove_urls(text):
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    return text

# Apply the function to the 'clean_review' column
df["clean_review"] = df["clean_review"].apply(remove_urls)

df[["review", "clean_review"]].sample(5)

,review,clean_review
3,I gave this app a chance it works up until the...,i gave this app a chance it works up until the...
3745,bagus dan lembab. TQ,bagus dan lembab. tq
404,Great 😃😃👍,great 😃😃👍
541,MAE by Mybank good 👍👌 Thank you so much,mae by mybank good 👍👌 thank you so much
3981,I can't view my transaction receipt,i can't view my transaction receipt


Remove HTML Tags from text

In [17]:
# Function to remove HTML tags from text
def remove_html(text):
    text = re.sub(r"<.*?>", "", text)
    return text

# Apply the function to the 'clean_review' column
df["clean_review"] = df["clean_review"].apply(remove_html)

df[["review", "clean_review"]].sample(5)

,review,clean_review
1767,Makin ke payah bodo je,makin ke payah bodo je
2365,apps bagus boleh bagi saya duit...duit sekaran...,apps bagus boleh bagi saya duit...duit sekaran...
3374,Apps ni tak boleh nak di buka start dari awal ...,apps ni tak boleh nak di buka start dari awal ...
1385,not that good..,not that good..
1912,i cant verify my account why must i waste my e...,i cant verify my account why must i waste my e...


Remove Emoji

In [18]:
# Function to remove emojis from text
def remove_emojis(text):
    return emoji.replace_emoji(text, replace="")

# Apply the function to the 'clean_review' column
df["clean_review"] = df["clean_review"].apply(remove_emojis)

df[["review", "clean_review"]].sample(5)

,review,clean_review
984,masalah,masalah
2497,Lambat dan lag bila nak cepat tpi bila xde apa...,lambat dan lag bila nak cepat tpi bila xde apa...
534,exellent,exellent
181,now very good 👍🏻😊,now very good
2586,selalu problem,selalu problem


Remove Punctuation

In [19]:
# Function to remove punctuation from text
def remove_punctuation(text):
    return text.translate(str.maketrans("", "", string.punctuation))

# Apply the function to the 'clean_review' column
df["clean_review"] = df["clean_review"].apply(remove_punctuation)

df[["review", "clean_review"]].sample(5)

,review,clean_review
3256,memudahkan pengguna,memudahkan pengguna
3513,Everytime update update.,everytime update update
3847,Developer sial.Dah 8 bulan aku bersabar dengan...,developer sialdah 8 bulan aku bersabar dengan ...
2210,before this can use but the latest update rest...,before this can use but the latest update rest...
3824,"Assalamualaikum en, nk tnya. Awat otp saya xdi...",assalamualaikum en nk tnya awat otp saya xdite...


Remove Extra Space

In [25]:
# Function to remove extra spaces from text
def remove_extra_spaces(text):
    return " ".join(text.split())

# Apply the function to the 'clean_review' column
df["clean_review"] = df["clean_review"].apply(remove_extra_spaces)

df[["review", "clean_review"]].sample(5)

,review,clean_review
2179,"I can't open this app anymore, because of OEM ...",i cant open this app anymore because of oem un...
3984,I made a transfer and the app shows that the t...,i made a transfer and the app shows that the t...
932,Mybank2u practical banking app and it is easy ...,mybanku practical banking app and it is easy t...
1667,5 ⭐,
1284,MAE or also known as Maybank2u is a very helpf...,mae or also known as maybanku is a very helpfu...


In [35]:
df[["review", "clean_review"]].sample(10)

,review,clean_review
2665,Respond to my issue really fast! Ty for helpin...,respond to my issue really fast ty for helping...
253,MAE is a convenient and user-friendly banking ...,mae is a convenient and userfriendly banking a...
2692,Good App🥰,good app
688,Reliable and secure banking app. Transactions ...,reliable and secure banking app transactions a...
2032,good.. intelligent..,good intelligent
2166,overlay apps is not hacking you Mae apps preve...,overlay apps is not hacking you mae apps preve...
3244,Saya suka,saya suka
3615,doesn't give verification to pay online . wast...,doesnt give verification to pay online waste m...
531,Kindly ask why I can't withdraw my money from ...,kindly ask why i cant withdraw my money from t...
3131,Blh lerrr,blh lerrr


Tokenization

In [36]:
# Function to tokenize text
def tokenize_text(text):
    return word_tokenize(text)

In [39]:
# Apply the function to the 'clean_review' column
df["tokens"] = df["clean_review"].apply(tokenize_text)

In [40]:
df[["clean_review", "tokens"]].head()

,clean_review,tokens
0,a dependable banking app that makes handling m...,"[a, dependable, banking, app, that, makes, han..."
1,very good app,"[very, good, app]"
2,amazing service and great experience thank you...,"[amazing, service, and, great, experience, tha..."
3,i gave this app a chance it works up until the...,"[i, gave, this, app, a, chance, it, works, up,..."
4,cant reload my phone credit through fpx notifi...,"[cant, reload, my, phone, credit, through, fpx..."


Stopword Removal

In [47]:
from nltk.corpus import stopwords

english_stopwords = set(stopwords.words("english"))

In [48]:
malay_stopwords = {
    "yang",
    "dan",
    "di",
    "ke",
    "dari",
    "untuk",
    "ini",
    "itu",
    "ni",
    "lah",
    "pun",
    "je",
    "jer",
    "dengan",
    "pada",
    "dalam",
    "oleh",
    "atau"
}

In [49]:
all_stopwords = english_stopwords.union(malay_stopwords)

# Function to remove stopwords from tokenized text
def remove_stopwords(tokens):
    return [
        word
        for word in tokens
        if word not in all_stopwords
    ]

# Apply the function to the 'tokens' column
df["tokens"] = df["tokens"].apply(remove_stopwords)

df[["clean_review", "tokens"]].head()

,clean_review,tokens
0,a dependable banking app that makes handling m...,"[dependable, banking, app, makes, handling, mo..."
1,very good app,"[good, app]"
2,amazing service and great experience thank you...,"[amazing, service, great, experience, thank, e..."
3,i gave this app a chance it works up until the...,"[gave, app, chance, works, restricted, apps, a..."
4,cant reload my phone credit through fpx notifi...,"[cant, reload, phone, credit, fpx, notificatio..."


Lemmatization

In [51]:
# Function to lemmatize tokens using spaCy
def lemmatize_tokens(tokens):
    text = " ".join(tokens)
    doc = nlp(text)

    return [token.lemma_ for token in doc]

# Apply the function to the 'tokens' column
df["tokens"] = df["tokens"].apply(lemmatize_tokens)

df["clean_review"] = df["tokens"].apply(lambda x: " ".join(x))

df[["review", "clean_review"]].head(10)

,review,clean_review
0,A dependable banking app that makes handling m...,dependable banking app make handle money prett...
1,very good app,good app
2,Amazing service and great experience! 😊 Thank ...,amazing service great experience thank excelle...
3,I gave this app a chance it works up until the...,give app chance work restrict app accessibilit...
4,"Can't reload my phone credit through FPX, noti...",can not reload phone credit fpx notification s...
5,if we have an award for most weak stupid nonse...,award weak stupid nonsense online banking woul...
6,very easy. i like it,easy like
7,Damn worst! Everytime I try to make Online pay...,damn worst everytime try make online payment t...
8,"two time qr payment timeout, but still deduct ...",two time qr payment timeout still deduct money...
9,Maybank2u offers secure online banking with co...,maybanku offer secure online banking convenien...


In [52]:
final_df = df[
    [
        "review_id",
        "review",
        "clean_review",
        "rating",
        "review_date",
        "app_version"
    ]
]

final_df.to_csv(
    "../data/cleaned_data.csv",
    index=False,
    encoding="utf-8-sig"
)